# Week 5

- Prior skal være på phi tilde og ikke phi 
- Skal være mellem -1 og 1
- Ar2 model istedet. 
- Højere orrdens markov processer. 

We use change of variable

$$
\phi = \frac{2e^{\tilde{\phi}}}{1 + e^{\tilde{\phi}}} - 1
$$

$$
\frac{d\phi}{d \tilde{\phi}} = (\frac{2e^{\tilde{\phi}}}{1 + e^{\tilde{\phi}}} )'

= 2e^{\tilde{\phi}} \cdot (1 + e^{\tilde{\phi}})^{-1}

= (2e^{\tilde{\phi}} )' \cdot (1 + e^{\tilde{\phi}})^{-1} + 2e^{\tilde{\phi}} \cdot ( (1 + e^{\tilde{\phi}})^{-1})'

$$

$$
2e^{\tilde{\phi}}  \cdot (1 + e^{\tilde{\phi}})^{-1} + 2e^{\tilde{\phi}} \cdot ( -1 (1 + e^{\tilde{\phi}})^{-2} \cdot e^{\tilde{\phi}} )
$$

$$
2e^{\tilde{\phi}}  \cdot (1 + e^{\tilde{\phi}})^{-1} -  2e^{2\tilde{\phi}} \cdot (1 + e^{\tilde{\phi}})^{-2} 
$$


$$
=
\frac{(1 + e^{\tilde{\phi}})2e^{\tilde{\phi}}}{  (1 + e^{\tilde{\phi}})^{-2} }-  \frac{2e^{2\tilde{\phi}}}{(1 + e^{\tilde{\phi}})^{-2} }
$$


$$
= \frac{2e^{\tilde{\phi}} + 2e^{2\tilde{\phi}} - 2e^{2\tilde{\phi}}}{(1 + e^{\tilde{\phi}})^{2}}
= \frac{2e^{\tilde{\phi}}}{(1 + e^{\tilde{\phi}})^{2}}
$$

## Config

In [1]:
import jax 
from src.data import load_model_and_data 
from src.optim.loss import negative_log_likelihood_v2
from src.config.modelnames import MODELTYPES 

jax.config.update("jax_enable_x64", True)  # Enable 64-bit precision for JAX 

PHI_SIGMA = 1
PHI_TILDE_SIGMA = 3


RUN_TO_LOAD = {
    "tag": "week_2",
    "run": 1
}




### Loading HMM Autoregressive models

In [ ]:
import jax.numpy as jnp

#Models 
from src.api.v1.ar_hmm import ArHMM
from src.api.v3.ar_prior_hmm_constrained import ArHMMPriorConstrained
from src.api.v3.ar_hmm_constrained import ArHMMConstrained
from src.api.v3.ar_prior_hmm import ArHMMPhiPrior


params, y_old, _ = load_model_and_data(modelname=MODELTYPES.AR_HMM, tag=RUN_TO_LOAD["tag"], run=RUN_TO_LOAD["run"])
y = y_old
#x = y_old[:-1]  # Previous observations as inputs
#y = y_old[1:]   # Current observations as targets
#X = jnp.stack([x], axis=1)  # Same lagged observations for each state, shape (T, num_states) 

print(params.mu)
## 

# Inverse of phi = (2*exp(phi_tilde))/(1+exp(phi_tilde)) - 1  =>  phi_tilde = log((1+phi)/(1-phi))
# Clip phi to (-1, 1) since the constrained model enforces this range
phi_clipped = jnp.clip(params.phi, -0.999, 0.999)
phi_tilde = jnp.log((1 + phi_clipped) / (1 - phi_clipped))

params 

[ 535.68046261  889.93105469 1306.70139423]


model: ArHMM
mu:    [ 535.68046261  889.93105469 1306.70139423]
sigma: [  0.4363331   39.538437   187.60279154  39.96637215]
phi:   [1.00005795 1.03787735 0.7414197  0.642594  ]
initial_state_dist:  None
transition_matrix:
[[5.17866542e-01 4.77790595e-01 4.34286321e-03 1.31017120e-17]
 [1.23282628e-01 7.07296253e-01 1.69421119e-01 5.50022377e-15]
 [3.97409097e-04 3.08985045e-01 5.82350886e-01 1.08266660e-01]
 [2.53832161e-13 3.46799955e-16 2.50925520e-01 7.49074480e-01]]

In [ ]:
import jax.numpy as jnp

#Models 
from src.api.v1.ar_hmm import ArHMM
from src.api.v3.ar_prior_hmm_constrained import ArHMMPriorConstrained
from src.api.v3.ar_hmm_constrained import ArHMMConstrained
from src.api.v3.ar_prior_hmm import ArHMMPhiPrior


params, y_old, _ = load_model_and_data(modelname=MODELTYPES.AR_HMM, tag=RUN_TO_LOAD["tag"], run=RUN_TO_LOAD["run"])
y = y_old
#x = y_old[:-1]  # Previous observations as inputs
#y = y_old[1:]   # Current observations as targets
#X = jnp.stack([x], axis=1)  # Same lagged observations for each state, shape (T, num_states) 

print(params.mu)
## 

# Inverse of phi = (2*exp(phi_tilde))/(1+exp(phi_tilde)) - 1  =>  phi_tilde = log((1+phi)/(1-phi))
# Clip phi to (-1, 1) since the constrained model enforces this range
phi_clipped = jnp.clip(params.phi, -0.999, 0.999)
phi_tilde = jnp.log((1 + phi_clipped) / (1 - phi_clipped))


phi_tilde = jnp.array([0.1,0.6,0,0.3])


model_free = ArHMM(transition_logits=params.transition_logits, mu=params.mu, log_sigma=jnp.log(params.sigma), phi=params.phi)

params.mu = jnp.concatenate([jnp.array([400.0]), params.mu])  # shape (num_states,) with fixed first state mean

print(params.mu)
model_constrained = ArHMMConstrained(transition_logits=params.transition_logits, mu=params.mu, log_sigma=jnp.log(params.sigma), phi_tilde=phi_tilde, lags=1) 
model_prior = ArHMMPhiPrior(transition_logits=params.transition_logits, mu=params.mu, log_sigma=jnp.log(params.sigma), phi=params.phi, phi_sigma=PHI_SIGMA, lags=1)
model_prior_constrained = ArHMMPriorConstrained(transition_logits=params.transition_logits, mu=params.mu, log_sigma=jnp.log(params.sigma), phi_tilde=phi_tilde, phi_sigma=PHI_TILDE_SIGMA, lags=1)
model_prior_constrained_2 = ArHMMPriorConstrained(transition_logits=params.transition_logits, mu=params.mu, log_sigma=jnp.log(params.sigma), phi_tilde=jnp.stack([phi_tilde] * 2).T, phi_sigma=PHI_TILDE_SIGMA, lags=2)

models_list = [ model_constrained, model_prior, model_prior_constrained, model_prior_constrained_2] 
#data_list = [(y, X)] * len(models_list)  # Same data for all models

[ 535.68046261  889.93105469 1306.70139423]
[ 400.          535.68046261  889.93105469 1306.70139423]


## Preforming Optimization

In [4]:
for i, model in enumerate(models_list):
    nll = negative_log_likelihood_v2(model, y, None)
    print(f"Model: {model.__class__.__name__}, Negative Log-Likelihood: {nll}") 


Model: ArHMMConstrained, Negative Log-Likelihood: 11924.127959642834
Model: ArHMMPhiPrior, Negative Log-Likelihood: 9143.81365588073
Model: ArHMMPriorConstrained, Negative Log-Likelihood: 11932.204247687583
Model: ArHMMPriorConstrained, Negative Log-Likelihood: 11374.516144576213


Preforming optimization

In [5]:
from src.optim.loss import negative_log_likelihood_v2
from src.optim.lbfgs import LFBGSOptimizer 
from src.optim.minimizer import Minimizer
from src.data import save_model_and_data

opt_model_list = [] 
opt_models_names = []

for model in models_list:
    #y, X = data_list[models_list.index(model)]  # Get corresponding data for the model  
    optimizer = LFBGSOptimizer(model=model, loss_fn=negative_log_likelihood_v2)
    optimized_model = optimizer.run(y)
    opt_model_list.append(optimized_model) 


    opt_models_names.append(optimized_model.__class__.__name__)
    print(f"Optimized model: {optimized_model.__class__.__name__}")
    print(f"Negative Log-Likelihood after optimization: {negative_log_likelihood_v2(optimized_model, y)}") 
    if hasattr(optimized_model, "log_prior"):
        print(f"Log Prior after optimization: {optimized_model.log_prior()}") 

    #Printing mu vals 
    if hasattr(optimized_model.emission, "_compute_mu"):
        print(f"Optimized mu values: {optimized_model.emission._compute_mu()}") 
    
    print("-" * 50)  # Separator for readability

    save_model_and_data(modelname = optimized_model.__class__.__name__, tag="week5", run=0, model=optimized_model, y=y)



Optimized model: ArHMMConstrained
Negative Log-Likelihood after optimization: 9611.153665863514
Optimized mu values: [3.99416454e+02 3.99420543e+02 1.12508857e+03 4.50284103e+05]
--------------------------------------------------
Optimized model: ArHMMPhiPrior
Negative Log-Likelihood after optimization: 9062.613175569357
Log Prior after optimization: -5.189054175706976
Optimized mu values: [ 400.00292065  619.59285547  619.71535983 1467.81326775]
--------------------------------------------------
Optimized model: ArHMMPriorConstrained
Negative Log-Likelihood after optimization: 9672.034214439735
Log Prior after optimization: -8.222957167260718
Optimized mu values: [403.37022171 403.55327481 942.20492857 942.20492857]
--------------------------------------------------
Optimized model: ArHMMPriorConstrained
Negative Log-Likelihood after optimization: 9383.583299643924
Log Prior after optimization: -16.315731588434844
Optimized mu values: [ 396.14670787  606.08852647  614.88668382 1426.18

In [6]:
optimized_model.emission.mu_diff

Array([396.14670787,   5.34683044,   2.17454231,   6.69864065], dtype=float64)

In [7]:
#printing phi vals 
for model in opt_model_list:
    print(f"Model: {model.__class__.__name__}")
    if (hasattr(model.emission, 'phi')):
        print(model.emission.phi)
    else: 
        phi = (2 * jnp.exp(model.emission.phi_tilde)) / (1 + jnp.exp(model.emission.phi_tilde)) - 1
        print(phi) 

Model: ArHMMConstrained
[0.26159272 1.         0.94335712 0.97602341]
Model: ArHMMPhiPrior
[1.0000934  1.04674216 0.62597201 0.73410024]
Model: ArHMMPriorConstrained
[0.03216303 1.         0.8667268  0.99865912]
Model: ArHMMPriorConstrained
[[ 0.34922716  0.03284886]
 [ 0.99853947  0.04501197]
 [ 0.98627214 -0.36344976]
 [ 0.94888433 -0.16880314]]


In [8]:
from statsmodels.tsa.arima.model import ARIMA
import numpy as np
y_np = np.array(y_old)  # Convert JAX array to NumPy array 


model = ARIMA(y_np, order=(2, 0, 0)).fit()

print(f"Log-Likelihood of AR(2) model: {model.llf:.4f}") 
print(model.summary())  

Log-Likelihood of AR(2) model: -10207.1111
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 1666
Model:                 ARIMA(2, 0, 0)   Log Likelihood              -10207.111
Date:                Mon, 06 Apr 2026   AIC                          20422.222
Time:                        09:11:47   BIC                          20443.895
Sample:                             0   HQIC                         20430.254
                               - 1666                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        805.6580     48.949     16.459      0.000     709.719     901.597
ar.L1          1.3033      0.015     84.654      0.000       1.273       1.334
ar.L2    

In [71]:
import equinox as eqx
import numpy as np
import matplotlib.pyplot as plt
from src.utils import plot_likelihood
from concurrent.futures import ThreadPoolExecutor

# phi[0] grid: constrained models only support (-1, 1), free models allow > 1
PHI_GRID_CONSTRAINED = np.linspace(0.50, 0.999, 50)
PHI_GRID_FREE        = np.linspace(0, 0.1,  50)
NUM_WORKERS = 8


def make_profile_loss(model, phi0):
    """Pin phi[0] to phi0, handling both phi and phi_tilde parameterisations."""
    if hasattr(model.emission, 'mu'):
        phi_tilde_0 = float(jnp.log((1 + phi0) / (1 - phi0)))
        def loss(m, y):
            pt = m.emission.mu.at[0].set(phi_tilde_0)
            m2 = eqx.tree_at(lambda z: z.emission.phi_tilde, m, pt)
            return negative_log_likelihood_v2(m2, y)
    else:
        def loss(m, y):
            p = m.emission.phi.at[0].set(phi0)
            m2 = eqx.tree_at(lambda z: z.emission.phi, m, p)
            return negative_log_likelihood_v2(m2, y)
    return loss


def profile_one_point(args):
    model, phi0, y = args
    loss_fn = make_profile_loss(model, phi0)
    opt = LFBGSOptimizer(model=model, loss_fn=loss_fn)
    opt_model = opt.run(y)
    return float(-loss_fn(opt_model, y)) 



#for model in opt_model_list:
#    print(f"Model: {model.__class__.__name__}")
#    is_constrained = hasattr(model.emission, 'phi_tilde')
#
#    phi_grid = PHI_GRID_CONSTRAINED if is_constrained else PHI_GRID_FREE                                                 
#
#    task_args = [(model, float(phi0), y, X) for phi0 in phi_grid]                                                              
#
#    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:                                                              
#        log_likes = np.array(list(executor.map(profile_one_point, task_args))) 
#
#
#    plot_likelihood(phi_grid, log_likes) 


In [73]:
model = model_prior_constrained


print(f"Model: {model.__class__.__name__}")
is_constrained = hasattr(model.emission, 'phi_tilde')

phi_grid = PHI_GRID_CONSTRAINED if is_constrained else PHI_GRID_FREE                                                 

task_args = [(model, float(phi0), y) for phi0 in phi_grid]                                                              

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:                                                              
    log_likes = np.array(list(executor.map(profile_one_point, task_args))) 


plot_likelihood(phi_grid, log_likes, zoom=0.5) 

Model: ArHMMPriorConstrained


TypeError: make_profile_loss.<locals>.loss() takes 2 positional arguments but 3 were given